In [1]:
!pip install chromadb sentence-transformers python-dotenv rich -q

In [2]:
import re
from pathlib import Path
from typing import List, Dict
import chromadb
import sentence_transformers

print("Imports successful")

Imports successful


In [3]:
CORPUS_DIR = Path("corpus/zoning")

MAX_CHARS = 2400   # ~600 tokens approximation
MIN_CHARS = 50

In [36]:
def load_markdown_files(corpus_dir: Path) -> List[Dict]:

    documents = []

    for file_path in corpus_dir.glob("*.md"):

        with open(file_path, "r", encoding="utf-8") as f:

            lines = f.readlines()

            text = "".join(lines)


        documents.append({
            "source_file": file_path.name,
            "text": text,
            "lines": lines
        })

    return documents

In [5]:
def parse_header(text: str) -> Dict:

    lines = text.split("\n")

    document_title = ""
    last_amended = ""

    for line in lines[:10]:

        line = line.strip()

        # Extract title
        if line.startswith("# "):
            document_title = line.replace("# ", "").strip()

        # Extract amendment date
        if "**Last Amended:**" in line:
            last_amended = (
                line.replace("**Last Amended:**", "")
                .strip()
            )

    return {
        "document_title": document_title,
        "last_amended": last_amended
    }

In [6]:
def split_into_sections(
    lines: List[str]
) -> List[Dict]:

    sections = []

    current_section = None

    current_content = []

    start_line = None

    for idx, line in enumerate(lines, start=1):

        # ---------------------------------
        # Detect new section
        # ---------------------------------

        if line.strip().startswith("## "):

            # Save previous section
            if current_section is not None:

                sections.append({

                    "section_title": current_section,

                    "content": "".join(
                        current_content
                    ).strip(),

                    "start_line": start_line,

                    "end_line": idx - 1
                })

            # Start new section
            current_section = (
                line.strip()
                .replace("## ", "")
                .strip()
            )

            current_content = []

            start_line = idx

        else:

            # Accumulate section content
            if current_section is not None:

                current_content.append(line)

    # ---------------------------------
    # Save final section
    # ---------------------------------

    if current_section is not None:

        sections.append({

            "section_title": current_section,

            "content": "".join(
                current_content
            ).strip(),

            "start_line": start_line,

            "end_line": len(lines)
        })

    return sections

In [7]:
# def split_large_section(
#     text: str,
#     max_chars: int = MAX_CHARS
# ) -> List[str]:

#     if len(text) <= max_chars:
#         return [text]

#     paragraphs = text.split("\n\n")

#     chunks = []
#     current_chunk = ""

#     for para in paragraphs:

#         para = para.strip()

#         if not para:
#             continue

#         candidate = current_chunk + "\n\n" + para

#         if len(candidate) > max_chars:

#             if current_chunk.strip():
#                 chunks.append(current_chunk.strip())

#             current_chunk = para

#         else:
#             current_chunk = candidate

#     if current_chunk.strip():
#         chunks.append(current_chunk.strip())

#     return chunks

In [8]:
def split_large_section(
    text: str,
    max_chars: int = MAX_CHARS
) -> List[str]:

    if len(text) <= max_chars:
        return [text]

    subsection_pattern = r"(?=^### )"

    subsections = re.split(
        subsection_pattern,
        text,
        flags=re.MULTILINE
    )

    subsections = [
        s.strip()
        for s in subsections
        if s.strip()
    ]

    chunks = []

    current_chunk = ""

    for subsection in subsections:

        candidate = (
            current_chunk
            + "\n\n"
            + subsection
        )

        if len(candidate) > max_chars:

            if current_chunk.strip():
                chunks.append(
                    current_chunk.strip()
                )

            current_chunk = subsection

        else:
            current_chunk = candidate

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [9]:
def build_chunks(
    document: Dict
) -> List[Dict]:

    source_file = document["source_file"]

    text = document["text"]

    lines = document["lines"]

    header = parse_header(text)

    # ---------------------------------
    # Line-aware section splitting
    # ---------------------------------

    sections = split_into_sections(
        lines
    )

    chunks = []

    chunk_index = 0

    for section in sections:

        section_title = section[
            "section_title"
        ]

        section_content = section[
            "content"
        ]

        start_line = section[
            "start_line"
        ]

        end_line = section[
            "end_line"
        ]

        split_chunks = split_large_section(
            section_content
        )

        # ---------------------------------
        # Approximate chunk line ranges
        # ---------------------------------

        total_chunks = len(split_chunks)

        total_lines = (
            end_line - start_line + 1
        )

        approx_lines_per_chunk = max(
            1,
            total_lines // total_chunks
        )

        for idx, chunk_text in enumerate(
            split_chunks
        ):

            chunk_text = chunk_text.strip()

            if len(chunk_text) < MIN_CHARS:
                continue

            # ---------------------------------
            # Remove redundant markdown metadata
            # ---------------------------------

            chunk_text = re.sub(
                r"\*\*Source:\*\*.*?\n",
                "",
                chunk_text
            )

            chunk_text = re.sub(
                r"\*\*Last Amended:\*\*.*?\n",
                "",
                chunk_text
            )

            chunk_text = re.sub(
                r"\*\*Retrieved:\*\*.*?\n",
                "",
                chunk_text
            )

            # ---------------------------------
            # Approximate provenance lines
            # ---------------------------------

            chunk_start_line = (
                start_line
                + (idx * approx_lines_per_chunk)
            )

            chunk_end_line = min(

                end_line,

                chunk_start_line
                + approx_lines_per_chunk
                - 1
            )

            # ---------------------------------
            # Prepend visible metadata
            # ---------------------------------

            prepended_text = (

                f"[Source: {source_file} | "
                f"Section: {section_title} | "
                f"Amended: {header['last_amended']} | "
                f"Lines: {chunk_start_line}-{chunk_end_line}]"

                f"\n\n{chunk_text}"
            )

            has_cross_ref = bool(

                re.search(
                    r"Section\s+\d{2}-\d{2,3}",
                    chunk_text
                )
            )

            chunks.append({

                "id":
                f"{source_file}_{chunk_index}",

                "document":
                prepended_text,

                "metadata": {

                    "source_file":
                    source_file,

                    "section_title":
                    section_title,

                    "last_amended":
                    header["last_amended"],

                    "chunk_index":
                    chunk_index,

                    "has_cross_ref":
                    has_cross_ref,

                    # NEW
                    "start_line":
                    chunk_start_line,

                    "end_line":
                    chunk_end_line
                }
            })

            chunk_index += 1

    return chunks

In [39]:
documents = load_markdown_files(CORPUS_DIR)

all_chunks = []

for document in documents:

    chunks = build_chunks(document)

    all_chunks.extend(chunks)

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 17


In [11]:
all_chunks[0]

{'id': 'zr_01_rules_of_construction.md_0',
 'document': '[Source: zr_01_rules_of_construction.md | Section: Section 12-01: Rules Applying to Text of Resolution | Amended: 2/2/2011 | Lines: 2-32]\n\n\nThe following rules of construction apply to the text of this Resolution:\n\n(a) The particular shall control the general.\n\n(b) In case of any difference of meaning or implication between the text of this Resolution and any caption, illustration, summary table or illustrative table, the text shall control.\n\n(c) The word "shall" is always mandatory and not discretionary. The word "may" is permissive.\n\n(d) Words used in the present tense shall include the future; and words used in the singular number shall include the plural, and the plural the singular, unless the context clearly indicates the contrary.\n\n(e) A "building" or "structure" includes any part thereof. The terms **residential building**, **commercial building** and **community facility building** shall refer to an entire *

In [12]:
for chunk in all_chunks[:5]:

    print("=" * 80)

    print(chunk["id"])

    print(len(chunk["document"]))

zr_01_rules_of_construction.md_0
2346
zr_01_rules_of_construction.md_1
1163
zr_02_definitions_key.md_0
2279
zr_02_definitions_key.md_1
1608
zr_03_rear_yard_requirements.md_0
1837


In [13]:
import chromadb

from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

In [14]:
CHROMA_PATH = "chroma_db"

COLLECTION_NAME = "zoning_docs"

In [15]:
client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [16]:
embedding_function = (
    SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)

In [17]:
existing_collections = client.list_collections()

existing_names = [
    collection.name
    for collection in existing_collections
]

if COLLECTION_NAME in existing_names:

    client.delete_collection(
        name=COLLECTION_NAME
    )

    print(f"Deleted existing collection: {COLLECTION_NAME}")

In [18]:
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

print("Collection created successfully")

Collection created successfully


In [19]:
documents = [
    chunk["document"]
    for chunk in all_chunks
]

metadatas = [
    chunk["metadata"]
    for chunk in all_chunks
]

ids = [
    chunk["id"]
    for chunk in all_chunks
]

In [20]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("Documents indexed successfully")

Documents indexed successfully


In [21]:
count = collection.count()

print(f"Total indexed documents: {count}")

Total indexed documents: 17


In [22]:
results = collection.query(
    query_texts=[
        "rear yard requirements in R6 districts"
    ],
    n_results=3
)

results

{'ids': [['zr_03_rear_yard_requirements.md_0',
   'zr_03_rear_yard_requirements.md_1',
   'zr_07_front_yard_requirements.md_0']],
 'embeddings': None,
 'documents': [['[Source: zr_03_rear_yard_requirements.md | Section: Section 23-34: Rear Yard and Rear Yard Equivalent Requirements | Amended: 5/12/2021 | Lines: 2-23]\n\n\n---\n\n### 23-342: Rear Yard Requirements\n\n**Applicable Districts:** R1 through R10\n\nIn all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).\n\nThe minimum required **rear yard** depth shall be:\n\n| District | Minimum Rear Yard Depth |\n|---|---|\n| R1 through R5 | 30 feet |\n| R6 through R10 | 30 feet |\n\nFor corner lots in any Residence District, no rear yard shall be required.\n\nFor through lots in any Residence District, eac

# Cross Reference Map Generation

In [23]:
import json

In [24]:
SECTION_ID_PATTERN = r"(\d{2}-\d{2,3})"

In [25]:
present_sections = set()

for chunk in all_chunks:

    # ---------------------------------
    # Extract from section title
    # ---------------------------------
    section_title = (
        chunk["metadata"]["section_title"]
    )

    title_matches = re.findall(
        SECTION_ID_PATTERN,
        section_title
    )

    for match in title_matches:
        present_sections.add(match)

    # ---------------------------------
    # Extract subsection IDs
    # Example:
    # ### 23-342:
    # ---------------------------------
    text = chunk["document"]

    subsection_matches = re.findall(
        r"###\s+(\d{2}-\d{2,3})",
        text
    )

    for match in subsection_matches:
        present_sections.add(match)

In [26]:
CROSS_REF_PATTERN = r"Section\s+(\d{2}-\d{2,3})"

In [27]:
referenced_sections = set()

for chunk in all_chunks:

    text = chunk["document"]

    matches = re.findall(
        CROSS_REF_PATTERN,
        text
    )

    for match in matches:
        referenced_sections.add(match)

In [28]:
referenced_but_missing = (
    referenced_sections
    - present_sections
)

In [29]:
cross_ref_map = {

    "present": sorted(
        list(present_sections)
    ),

    "referenced_but_missing": sorted(
        list(referenced_but_missing)
    )
}
# cross_ref_map

In [30]:
cross_ref_path = (
    Path(CHROMA_PATH)
    / "cross_ref_map.json"
)

with open(
    cross_ref_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cross_ref_map,
        f,
        indent=2
    )

print(f"Saved: {cross_ref_path}")

Saved: chroma_db\cross_ref_map.json


In [31]:
print(
    f"Present section IDs: "
    f"{len(present_sections)}"
)

print(
    f"Referenced but missing: "
    f"{len(referenced_but_missing)}"
)

Present section IDs: 17
Referenced but missing: 7


In [32]:
from rich.console import Console
from rich.table import Table

In [33]:
console = Console()

In [34]:
file_stats = []

In [40]:

for document in documents:

    source_file = document["source_file"]

    header = parse_header(
        document["text"]
    )

    chunks = build_chunks(document)

    file_stats.append({

        "source_file": source_file,

        "last_amended": (
            header["last_amended"]
        ),

        "chunks_created": len(chunks)
    })

In [41]:
table = Table(
    title="Zoning Corpus Ingest Summary"
)

table.add_column(
    "Filename",
    style="cyan"
)

table.add_column(
    "Chunks Created",
    justify="right",
    style="green"
)

table.add_column(
    "Last Amended",
    style="yellow"
)

In [42]:
for stat in file_stats:

    table.add_row(

        stat["source_file"],

        str(stat["chunks_created"]),

        stat["last_amended"]
    )

In [43]:
console.print(table)

                         Zoning Corpus Ingest Summary                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Filename                                   ┃ Chunks Created ┃ Last Amended ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ zr_01_rules_of_construction.md             │              2 │ 2/2/2011     │
│ zr_02_definitions_key.md                   │              2 │ 8/14/2025    │
│ zr_03_rear_yard_requirements.md            │              2 │ 5/12/2021    │
│ zr_04_permitted_obstructions_rear_yard.md  │              1 │ 11/10/2022   │
│ zr_05_floor_area_R6_R12_current.md         │              1 │ 4/30/2024    │
│ zr_06_floor_area_R6_R10_SUPERSEDED_2019.md │              1 │              │
│ zr_07_front_yard_requirements.md           │              1 │ 6/3/2020     │
│ zr_08_permitted_obstructions_all_yards.md  │              1 │ 11/10/2022   │
│ zr_09_ceqr_e_designations.md               │              4 │              │
│ zr_10_height_setback_R6_R12.md             │              2 │ 4/30/2024    │
└────────────────────────────────────────────┴────────────────┴──────────────┘

In [44]:
console.print(
    f"\n[bold green]Total chunks indexed:[/bold green] "
    f"{len(all_chunks)}"
)

Total chunks indexed: 17

In [45]:
console.print(
    f"[bold blue]Present section IDs:[/bold blue] "
    f"{len(present_sections)}"
)

console.print(
    f"[bold red]Referenced but missing:[/bold red] "
    f"{len(referenced_but_missing)}"
)

Present section IDs: 17

Referenced but missing: 7

In [46]:
console.print(
    f"\n[bold yellow]ChromaDB persisted at:[/bold yellow] "
    f"{CHROMA_PATH}"
)

ChromaDB persisted at: chroma_db